In [1]:
import pandas as pd
import numpy as np

In [2]:
#read the csv from my data sets file
df = pd.read_csv("../data/clean/AAPL_clean.csv")

In [3]:
df

,Date,Close,High,Low,Open,Volume,Return,Market_Return,Excess_Return,Corr_30,Beta_30
0,2020-01-02,72.400497,72.460761,71.156659,71.409763,135480400,NaN,NaN,NaN,NaN,NaN
1,2020-01-03,71.696632,72.455950,71.472454,71.629138,146322800,-0.009722,-0.007572,-0.002150,NaN,NaN
2,2020-01-06,72.267937,72.306506,70.568510,70.819208,118387200,0.007968,0.003815,0.004153,NaN,NaN
3,2020-01-07,71.928078,72.533118,71.708718,72.277601,108872000,-0.004703,-0.002812,-0.001891,NaN,NaN
4,2020-01-08,73.085114,73.386431,71.631559,71.631559,132079200,0.016086,0.005330,0.010756,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1253,2024-12-24,256.797180,256.807105,253.902972,254.101897,23234700,0.011478,0.011115,0.000363,0.595755,0.657770
1254,2024-12-26,257.612732,258.686881,256.230300,256.787255,27237100,0.003176,0.000067,0.003109,0.592286,0.653013
1255,2024-12-27,254.201355,257.294474,251.685102,256.429175,42355300,-0.013242,-0.010526,-0.002716,0.624768,0.715141
1256,2024-12-30,250.829788,252.122728,249.387669,250.859624,35557500,-0.013263,-0.011412,-0.001852,0.700883,0.814335


In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [5]:
# make sure the date is correct
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'])
else:
    df = df.reset_index()
    df['Date'] = pd.to_datetime(df['Date'])

In [6]:
# culculate moving averages
df['MA20'] = df['Close'].rolling(window=20).mean()
df['MA50'] = df['Close'].rolling(window=50).mean()

In [7]:
# create the figure for price trend
fig_price = go.Figure()

# 1. K candlestick
fig_price.add_trace(go.Candlestick(
    x=df['Date'], open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'],
    name='K Line',
    increasing_line_color='#26a69a', decreasing_line_color='#ef5350'
))

# 2. draw MA20 and MA50
fig_price.add_trace(go.Scatter(
    x=df['Date'], y=df['MA20'], line=dict(color='orange', width=1.5), name='MA 20'
))
fig_price.add_trace(go.Scatter(
    x=df['Date'], y=df['MA50'], line=dict(color='#29b6f6', width=1.5), name='MA 50'
))

# layout settings
fig_price.update_layout(
    title='Stock Price Trend (Candlestick + MA)',
    yaxis_title='Price (USD)',
    xaxis_title='Date',
    xaxis_rangeslider_visible=True, # time slider
    template='plotly_dark',
    height=500
)

# remove weekends
fig_price.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])]) 

fig_price.show()


In [8]:
# define the color green for increasing and red for decreasing
colors = ['#26a69a' if close >= open else '#ef5350' 
          for close, open in zip(df['Close'], df['Open'])]

In [9]:
# create the figure for volume analysis
fig_vol = go.Figure()

# draw volume bars with colors based on price movement
fig_vol.add_trace(go.Bar(
    x=df['Date'], 
    y=df['Volume'],
    marker_color=colors, 
    name='Volume'
))


fig_vol.update_layout(
    title='Daily Trading Volume Analysis',
    yaxis_title='Volume',
    xaxis_title='Date',
    xaxis_rangeslider_visible=True, # time slider
    height=500 
)

# delete data at weekend
fig_vol.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])]) 

fig_vol.show()

In [10]:
# 1. culculate cumulative return
df['Stock_Cum_Ret'] = (1 + df['Return'].fillna(0)).cumprod() - 1
df['Market_Cum_Ret'] = (1 + df['Market_Return'].fillna(0)).cumprod() - 1

# 2. create figure
fig = go.Figure()

# 3. add stocks cumulative return line
fig.add_trace(go.Scatter(
    x=df['Date'], 
    y=df['Stock_Cum_Ret'],
    mode='lines',
    name='Strategy / Stock Return',
    line=dict(color='#00E676', width=2.5) 
))

# 4. add market cumulative return line 
fig.add_trace(go.Scatter(
    x=df['Date'], 
    y=df['Market_Cum_Ret'],
    mode='lines',
    name='Market (SPY) Return',
    line=dict(color='#B0BEC5', width=2)
))

fig.update_layout(
    title='Cumulative Returns: Stock vs. Market',
    xaxis_title='Date',
    yaxis_title='Cumulative Return',
    yaxis=dict(tickformat='.0%'),
    template='plotly_dark',
    hovermode='x unified',        
    legend=dict(
            yanchor="top", y=0.99, xanchor="left", x=0.01 
        )
)

# show the figure
fig.show()

In [11]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['Date'], 
    y=df['Corr_30'],
    mode='lines',
    name='30-Day Correlation',
    line=dict(color='#FFCA28', width=2) 
))


fig.add_hline(
    y=0, 
    line_dash="dash", 
    line_color="#EF5350", 
    annotation_text="Decoupling Point (Zero Correlation)", 
    annotation_position="bottom right"
)

fig.add_hline(
    y=0.5, 
    line_dash="dot", 
    line_color="gray", 
)


fig.update_layout(
    title='Stock-Market Decoupling Analysis: 30-Day Rolling Correlation',
    xaxis_title='Date',
    yaxis_title='Pearson Correlation (r)',
    yaxis=dict(range=[-1, 1]), # 
    template='plotly_dark',
    hovermode='x unified'
)

fig.show()

In [13]:
import plotly.express as px

features_to_plot = ['Return', 'Market_Return', 'Excess_Return', 'Volume', 'Corr_30', 'Beta_30']
corr_matrix = df[features_to_plot].corr()

fig_corr = px.imshow(
    corr_matrix, 
    text_auto=".2f", 
    aspect="auto", 
    color_continuous_scale='RdBu_r', 
    title="Feature Correlation Heatmap"
)

fig_corr.update_layout(template='plotly_dark', width=700, height=600)
fig_corr.show()

In [16]:
'''
import plotly.express as px

fig_dist = px.histogram(
    df, 
    x="Return", 
    nbins=100,
    marginal="box", 
    color_discrete_sequence=['#00E676'], 
    title="Daily Return Distribution (Fat Tails Analysis)"
)

fig_dist.update_layout(
    xaxis_title='Daily Return',
    yaxis_title='Frequency (Days)',
    template='plotly_dark'
)

fig_dist.add_vline(x=0, line_dash="dash", line_color="red")

fig_dist.show()
'''

'\nimport plotly.express as px\n\nfig_dist = px.histogram(\n    df, \n    x="Return", \n    nbins=100,\n    marginal="box", \n    color_discrete_sequence=[\'#00E676\'], \n    title="Daily Return Distribution (Fat Tails Analysis)"\n)\n\nfig_dist.update_layout(\n    xaxis_title=\'Daily Return\',\n    yaxis_title=\'Frequency (Days)\',\n    template=\'plotly_dark\'\n)\n\nfig_dist.add_vline(x=0, line_dash="dash", line_color="red")\n\nfig_dist.show()\n'